# 02 — Entraînement du modèle voix (biomarqueurs tabulaires)

## Objectif
- Comprendre la **modalité voix** de ce projet : prédiction exploratoire à partir de **mesures vocales déjà calculées** (pas d’audio brut dans cette version).
- Comparer plusieurs modèles de **machine learning** sur le jeu **Oxford / UCI Parkinson**.
- Exporter un fichier `joblib` compatible avec Flask (`VoicePredictor`).

**Rappel important :** ce notebook et l’application web ne fournissent **pas** de diagnostic médical. Ce sont des outils pédagogiques et de démonstration.

## Audio brut vs biomarqueurs
- **Audio brut** : fichier son (ondes). Il faudrait extraire des descripteurs (MFCC, etc.) avant le ML.
- **Biomarqueurs tabulaires** (ici) : colonnes numériques (jitter, shimmer, etc.) déjà mesurées par les auteurs du dataset. Le modèle apprend une relation **nombre → risque exploratoire**.


## 1. Chargement du dataset

Le fichier `parkinsons.data` est une table CSV fournie avec le dataset Oxford (voir aussi `parkinsons.names`).

- Colonne **`status`** : `0` = sain (contrôle), `1` = Parkinson **dans ce jeu de données** (étiquette clinique telle qu’enregistrée à l’époque).
- Colonne **`name`** : identifiant d’enregistrement ; plusieurs lignes peuvent correspondre au même sujet.

**Glossaire — data leakage (fuite de données)** : utiliser accidentellement une information qui « triche » (par ex. mélanger des lignes du même patient entre train et test) donne une performance irréaliste. D’où la validation **par groupe**.


In [24]:
from pathlib import Path
import re
import json
import sys
import numpy as np
import pandas as pd
import joblib
from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

# Racine du projet Voice_Casin_Wayne_Oxford_AI (pour import src.modalities.voice.*)
_REPO_ROOT = Path("../..").resolve()
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

# Données : data/parkinsons.data à la racine de ce dossier projet
DATA_PATH = (_REPO_ROOT / "data" / "parkinsons.data").resolve()
assert DATA_PATH.exists(), f"Fichier introuvable : {DATA_PATH}"

df = pd.read_csv(DATA_PATH)
df.head()


,name,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,...,Shimmer:DDA,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE
0,phon_R01_S01_1,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,...,0.06545,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,phon_R01_S01_2,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,...,0.09403,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,phon_R01_S01_3,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,...,0.08270,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634
3,phon_R01_S01_4,116.676,137.871,111.366,0.00997,0.00009,0.00502,0.00698,0.01505,0.05492,...,0.08771,0.01353,20.644,1,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975
4,phon_R01_S01_5,116.014,141.781,110.655,0.01284,0.00011,0.00655,0.00908,0.01966,0.06425,...,0.10470,0.01767,19.649,1,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335


## 2. Compréhension des colonnes (familles)

| Famille | Exemples | Idée simple |
|---------|-----------|-------------|
| Fréquence fondamentale | `MDVP:Fo(Hz)`, `Fhi`, `Flo` | Hauteur / stabilité de la voix |
| **Jitter** | `MDVP:Jitter(%)`, `RAP`, `PPQ` | Petites variations **rapides** de la fréquence (instabilité) |
| **Shimmer** | `MDVP:Shimmer`, `APQ*` | Variations **d’amplitude** (volume) |
| Bruit harmonique | `NHR`, **`HNR`** | Rapport bruit / composante harmonique ; **HNR** élevé = voix plus « claire » |
| Non-linéaires | **`RPDE`**, **`DFA`**, `D2`, **`PPE`** | Mesures de complexité / régularité du signal (définitions mathématiques avancées — retenir : elles résument des motifs difficiles à voir à l’œil) |
| Cible | `status` | Ce que le modèle doit prédire (0/1) |


## 3. Data leakage et validation par patient

**Pourquoi éviter un split aléatoire simple ?**  
Les lignes ne sont **pas indépendantes** : un même patient a plusieurs enregistrements. Si des enregistrements du même patient sont à la fois en train et en test, le modèle peut apprendre l’identité du patient plutôt que la pathologie → **performance trop optimiste**.

**GroupKFold** : à chaque fold, les **groupes** (ici un identifiant patient approximatif dérivé de `name`) sont soit entièrement en train, soit entièrement en test.

**Glossaire — GroupKFold** : une validation croisée qui respecte des blocs (groupes) sans les couper entre train et test.


In [25]:
def patient_group(name: str) -> str:
    """Identifiant de groupe : enlève le suffixe numérique final (ex. phon_R01_S01_1 → phon_R01_S01)."""
    return re.sub(r"_\d+$", "", str(name))

df["patient_group"] = df["name"].map(patient_group)
df[["name", "patient_group", "status"]].head(8)


,name,patient_group,status
0,phon_R01_S01_1,phon_R01_S01,1
1,phon_R01_S01_2,phon_R01_S01,1
2,phon_R01_S01_3,phon_R01_S01,1
3,phon_R01_S01_4,phon_R01_S01,1
4,phon_R01_S01_5,phon_R01_S01,1
5,phon_R01_S01_6,phon_R01_S01,1
6,phon_R01_S02_1,phon_R01_S02,1
7,phon_R01_S02_2,phon_R01_S02,1


## 4. Vérification de la structure du dataset

Avant d'entraîner quoi que ce soit, on vérifie **proprement** la structure de `df`. Cela permet d'éviter une erreur classique : confondre le **nombre de lignes** (enregistrements vocaux) avec le **nombre de patients réels**, et donc sur-estimer la quantité de données indépendantes dont on dispose.

- Une **ligne** = un **enregistrement vocal** déjà résumé en 22 mesures numériques.
- Un **groupe-patient approximatif** est obtenu à partir de `name` en retirant le suffixe `_1`, `_2`, … (voir la fonction `patient_group` juste au-dessus). Plusieurs lignes pour le même groupe = plusieurs enregistrements du même sujet.
- Cette inspection permet de chiffrer trois choses qu'on doit savoir défendre à l'oral :
  1. combien de **lignes** au total,
  2. combien de **groupes-patients** uniques,
  3. la répartition **Parkinson (status = 1)** vs **contrôle (status = 0)** au niveau lignes **et** au niveau groupes.
- Elle vérifie aussi qu'**aucun groupe** ne mélange `status = 0` et `status = 1` : si c'était le cas, le `patient_group` serait trop large et casserait `GroupKFold`.
- C'est la justification concrète de l'usage de `GroupKFold` à la section suivante : **éviter la fuite de données** quand un même patient apparaît à la fois dans le train et dans le test.


In [26]:
print("=== STRUCTURE GÉNÉRALE DU DATASET ===")
print(f"Nombre total d'échantillons / enregistrements vocaux : {len(df)}")
print(f"Nombre total de groupes-patients approximatifs : {df['patient_group'].nunique()}")

print("\n=== RÉPARTITION DES ÉCHANTILLONS PAR CLASSE ===")
sample_counts = df["status"].value_counts().sort_index()
print(f"Contrôles sains (status = 0) : {sample_counts.get(0, 0)} échantillons")
print(f"Parkinson (status = 1)       : {sample_counts.get(1, 0)} échantillons")

print("\n=== RÉPARTITION DES GROUPES-PATIENTS PAR CLASSE ===")
group_status = df.groupby("patient_group")["status"].first()
group_counts = group_status.value_counts().sort_index()
print(f"Groupes contrôles sains (status = 0) : {group_counts.get(0, 0)} groupes")
print(f"Groupes Parkinson (status = 1)       : {group_counts.get(1, 0)} groupes")

print("\n=== NOMBRE D'ENREGISTREMENTS PAR GROUPE-PATIENT ===")
group_sizes = (
    df.groupby(["patient_group", "status"])
    .size()
    .reset_index(name="n_samples")
    .sort_values(["status", "patient_group"])
    .reset_index(drop=True)
)
display(group_sizes)

print("\n=== VÉRIFICATION DE COHÉRENCE DES STATUTS PAR GROUPE ===")
status_per_group = df.groupby("patient_group")["status"].nunique()
problematic_groups = status_per_group[status_per_group > 1]

if len(problematic_groups) == 0:
    print("OK : chaque groupe-patient possède un seul statut (pas de mélange Parkinson / contrôle).")
else:
    print("ATTENTION : certains groupes-patients ont plusieurs statuts différents.")
    display(problematic_groups)


=== STRUCTURE GÉNÉRALE DU DATASET ===
Nombre total d'échantillons / enregistrements vocaux : 195
Nombre total de groupes-patients approximatifs : 32

=== RÉPARTITION DES ÉCHANTILLONS PAR CLASSE ===
Contrôles sains (status = 0) : 48 échantillons
Parkinson (status = 1)       : 147 échantillons

=== RÉPARTITION DES GROUPES-PATIENTS PAR CLASSE ===
Groupes contrôles sains (status = 0) : 8 groupes
Groupes Parkinson (status = 1)       : 24 groupes

=== NOMBRE D'ENREGISTREMENTS PAR GROUPE-PATIENT ===


,patient_group,status,n_samples
0,phon_R01_S07,0,6
1,phon_R01_S10,0,6
2,phon_R01_S13,0,6
3,phon_R01_S17,0,6
4,phon_R01_S42,0,6
5,phon_R01_S43,0,6
6,phon_R01_S49,0,6
7,phon_R01_S50,0,6
8,phon_R01_S01,1,6
9,phon_R01_S02,1,6



=== VÉRIFICATION DE COHÉRENCE DES STATUTS PAR GROUPE ===
OK : chaque groupe-patient possède un seul statut (pas de mélange Parkinson / contrôle).


## 5. Préparation X, y, groups

- **X** : toutes les colonnes sauf `name`, `status` et colonnes auxiliaires.
- **y** : `status`.
- **groups** : `patient_group` pour le GroupKFold.

**Pipeline (glossaire)** : enchaîne des étapes (ex. normalisation + classifieur) pour appliquer exactement les mêmes transformations à l’entraînement et à la prédiction.


In [27]:
feature_cols = [c for c in df.columns if c not in ("name", "status", "patient_group")]
X = df[feature_cols].to_numpy(dtype=np.float64)
y = df["status"].to_numpy(dtype=np.int32)
groups = df["patient_group"].to_numpy()
feature_cols[:5], len(feature_cols)


(['MDVP:Fo(Hz)',
  'MDVP:Fhi(Hz)',
  'MDVP:Flo(Hz)',
  'MDVP:Jitter(%)',
  'MDVP:Jitter(Abs)'],
 22)

## 6. Modèles testés (intuition)

| Modèle | Rôle pédagogique |
|--------|------------------|
| **Régression logistique** | Baseline simple, coefficients interprétables. |
| **SVM RBF** | Frontière non linéaire ; attention au **surapprentissage** (*overfitting* : coller au bruit du train). |
| **Random Forest** | Ensemble d’arbres ; souvent robuste ; **feature importance**. |
| **ExtraTrees** | Comme RF mais plus aléatoire ; parfois très stable sur petits jeux. |
| **Gradient Boosting** | Arbres séquentiels qui corrigent les erreurs ; puissant mais à régulariser. |
| **XGBoost** | Implémentation efficace du gradient boosting (si le paquet `xgboost` est installé). |

**Ensemble learning** : combiner plusieurs modèles (ex. forêt = ensemble d’arbres). *VotingClassifier* n’est utilisé ici que si tu constates un gain clair (sinon complexité inutile).


In [28]:
def build_candidates():
    cands = [
        ("logistic_regression", Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=8000, class_weight="balanced", random_state=42)),
        ])),
        ("svc_rbf", Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42)),
        ])),
        ("random_forest", RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=42, n_jobs=-1)),
        ("extra_trees", ExtraTreesClassifier(n_estimators=500, class_weight="balanced", random_state=42, n_jobs=-1)),
        ("gradient_boosting", GradientBoostingClassifier(random_state=42)),
    ]
    try:
        from xgboost import XGBClassifier
        cands.append(("xgboost", Pipeline([
            ("scaler", StandardScaler()),
            ("clf", XGBClassifier(
                n_estimators=120, max_depth=3, learning_rate=0.06, subsample=0.9,
                colsample_bytree=0.9, reg_lambda=1.0, objective="binary:logistic",
                eval_metric="logloss", random_state=42, tree_method="hist",
            )),
        ])))
    except ImportError:
        print("xgboost non installé : étape ignorée.")
    return cands

candidates = build_candidates()
[name for name, _ in candidates]


['logistic_regression',
 'svc_rbf',
 'random_forest',
 'extra_trees',
 'gradient_boosting',
 'xgboost']

## 7. Métriques (phrases simples)

- **Accuracy** : proportion de prédictions correctes. **Limite** : si une classe est rare, on peut être « bon » en prédisant toujours la majorité.
- **Balanced accuracy** : moyenne des rappels par classe — plus juste si **déséquilibre**.
- **Recall Parkinson** : parmi les vrais Parkinson, combien le modèle retrouve (sensibilité côté malade).
- **Specificity / rappel contrôle** : capacité à reconnaître les contrôles.
- **ROC-AUC** : capacité à **ordonner** les individus (donner un score plus élevé aux Parkinson). Utile pour comparer des modèles qui sortent une **probabilité** (*probability score*).
- **F1-score** : compromis entre précision et rappel.

**Pourquoi un score élevé peut tromper ?**  
Surapprentissage, fuite de données, dataset trop petit, ou probabilités mal **calibrées** (le chiffre affiché n’est pas une vraie fréquence clinique).


## 5 bis. Augmentation tabulaire (SMOTE)

**SMOTE** crée des exemples **synthétiques** de la classe minoritaire en interpolant entre voisins dans l'espace des **features numériques**. Ce n'est pas de l'augmentation audio.

- **Sans fuite** : SMOTE s'applique uniquement sur les indices d'**apprentissage** de chaque pli ; le **test** reste composé de **vraies** lignes.
- **Risque** : sur un très petit jeu, les points synthétiques peuvent être peu réalistes ; cela ne remplace pas de nouveaux patients réels.

Le script `scripts/train_voice_tabular.py` (dans ce même dossier projet) propose `--augment-cv` et `--augment-final` (voir README à la racine).


In [29]:
from src.modalities.voice.augmentation import apply_smote_safe

USE_SMOTE_CV = False  # mets True pour comparer avec suréchantillonnage sur chaque train de pli

gkf = GroupKFold(n_splits=5)
leaderboard = []
for name, est in candidates:
    fold_aucs = []
    for tr, te in gkf.split(X, y, groups):
        x_tr, y_tr = X[tr], y[tr]
        if USE_SMOTE_CV:
            x_tr, y_tr, _ok = apply_smote_safe(x_tr, y_tr)
        m = clone(est)
        m.fit(x_tr, y_tr)
        if hasattr(m, "predict_proba"):
            p = m.predict_proba(X[te])[:, 1]
        else:
            d = m.decision_function(X[te])
            p = 1.0 / (1.0 + np.exp(-d))
        fold_aucs.append(roc_auc_score(y[te], p))
    leaderboard.append((name, float(np.mean(fold_aucs)), float(np.std(fold_aucs))))

leaderboard = sorted(leaderboard, key=lambda t: -(t[1] - 0.35 * t[2]))
pd.DataFrame(leaderboard, columns=["model", "mean_roc_auc", "std_roc_auc"]).assign(
    selection=lambda d: d["mean_roc_auc"] - 0.35 * d["std_roc_auc"]
)


,model,mean_roc_auc,std_roc_auc,selection
0,extra_trees,0.829211,0.142736,0.779254
1,random_forest,0.813964,0.172719,0.753512
2,xgboost,0.802966,0.210331,0.729350
3,gradient_boosting,0.778351,0.185653,0.713373
4,svc_rbf,0.736371,0.133600,0.689611
5,logistic_regression,0.720806,0.154927,0.666582


## 8. Choix du modèle final (compromis)

On ne retient **pas** aveuglément le meilleur AUC : on pénalise un peu la **variabilité** entre folds (`mean - 0.35 * std`).  
Exemple d’argument attendu : *« XGBoost a un AUC moyen élevé mais une forte variance → ExtraTrees est plus stable pour la démo. »* (Les chiffres dépendent de ton exécution.)

**Calibration** : ici le score est surtout un **score exploratoire** cohérent avec `score_to_label` côté Flask ; une vraie calibration clinique demanderait d’autres données et protocoles.


In [30]:
best_name, best_mean, best_std = leaderboard[0]
print(f"Choisi : {best_name} (mean ROC-AUC={best_mean:.3f}, std={best_std:.3f})")
best_est = dict(candidates)[best_name]
final = clone(best_est)
USE_SMOTE_FINAL = False  # True = même logique que --augment-final du script
from src.modalities.voice.augmentation import apply_smote_safe

x_fit, y_fit = X, y
if USE_SMOTE_FINAL:
    x_fit, y_fit, _ok = apply_smote_safe(X, y)
final.fit(x_fit, y_fit)


Choisi : extra_trees (mean ROC-AUC=0.829, std=0.143)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=FalseWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",False
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metr

## 9. Export joblib (contenu de l’artefact)

Le fichier contient typiquement :
- **`pipeline`** : modèle scikit-learn prêt pour `predict_proba`.
- **`features`** : ordre des colonnes attendu à l’inférence.
- **`threshold`** : seuil pour transformer le score en label `low/moderate/elevated` côté API.
- **`metrics`** : scores de comparaison pour la traçabilité.

**Pourquoi stocker `features` ?** Pour refuser ou aligner une requête JSON qui n’a pas les bonnes clés/valeurs.


In [31]:
MODEL_OUT = (_REPO_ROOT / "models" / "voice_parkinsons_tabular.joblib").resolve()
artifact = {
    "pipeline": final,
    "features": feature_cols,
    "model": best_name,
    "threshold": 0.5,
    "note": "Export depuis notebook 02_train_voice_model_conforme.ipynb",
    "metrics": {
        "leaderboard": [
            {"model": a, "mean_roc_auc": b, "std_roc_auc": c, "selection_score": b - 0.35 * c}
            for a, b, c in leaderboard
        ],
        "chosen": best_name,
    },
    "data_path_hint": str(DATA_PATH),
}
joblib.dump(artifact, MODEL_OUT)
print("Écrit:", MODEL_OUT)


Écrit: C:\Users\vikto\OneDrive - Haute Ecole Louvain en Hainaut\Documents\MA1\IA\IAthon\Voice_Casin_Wayne_Oxford_AI\models\voice_parkinsons_tabular.joblib


## 10. Smoke test — compatible PredictionResult

**Score** : probabilité estimée `P(status=1)` (Parkinson dans ce dataset).  
**Confidence** côté API Flask : confiance **technique** (qualité des données / prudence), **pas** une certitude médicale — le `VoicePredictor` la borne (ex. jamais 1.0).

Flask **n’importe jamais** ce notebook : il charge seulement le `joblib`.


In [32]:
loaded = joblib.load(MODEL_OUT)
row = X[:1]
p = float(loaded["pipeline"].predict_proba(row)[0, 1])
pseudo = {
    "modality": "voice",
    "status": "ok",
    "score": p,
    "confidence": 0.75,
    "label": "moderate",
    "details": {"model_name": loaded.get("model")},
    "warnings": [],
}
print(json.dumps(pseudo, indent=2))


{
  "modality": "voice",
  "status": "ok",
  "score": 1.0,
  "confidence": 0.75,
  "label": "moderate",
  "details": {
    "model_name": "extra_trees"
  },
  "warnings": []
}
